In [2]:
"""
LangGraph ReAct agent + MySQL tools (schema, NL->SQL, execute)
Compatible with:
  - langchain==1.0.3
  - langchain-community==0.4.1
  - langchain-openai==1.0.1
  - langgraph>=0.2.x
  - duckduckgo-search>=6.x

Env you need:
  MYSQL_HOST, MYSQL_PORT, MYSQL_USER, MYSQL_PASSWORD, MYSQL_DB
  OPENAI_API_KEY
  (optional) MYSQL_MAX_ROWS
"""
from dotenv import load_dotenv
load_dotenv()
import os
import time
from typing import Any, Optional, List
import re

# --- LangChain / LangGraph imports (v1-compatible) ---------------------------
from langchain_openai import ChatOpenAI
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.utilities.sql_database import SQLDatabase
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import BaseTool
from langchain_classic.chains import create_sql_query_chain  # v1.x re-export
from langgraph.prebuilt import create_react_agent

# --- SQLAlchemy for safe execution ------------------------------------------
from sqlalchemy import create_engine, text


# ====================== Helpers =============================================

def _mysql_uri_from_env() -> str:
    host = os.getenv("MYSQL_HOST", "localhost")
    port = int(os.getenv("MYSQL_PORT", "3306"))
    user = os.getenv("MYSQL_USER")
    pwd = os.getenv("MYSQL_PASSWORD")
    db  = os.getenv("MYSQL_DB")
    if not all([user, pwd, db]):
        raise RuntimeError("Set MYSQL_USER, MYSQL_PASSWORD, and MYSQL_DB in env")
    return f"mysql+pymysql://{user}:{pwd}@{host}:{port}/{db}"

def _get_db() -> SQLDatabase:
    # Use SQLDatabase for schema & dialect awareness
    return SQLDatabase.from_uri(_mysql_uri_from_env())

def _get_engine():
    # Execute with SQLAlchemy engine (avoids relying on SQLDatabase internals)
    return create_engine(_mysql_uri_from_env(), pool_pre_ping=True)

_SELECT_ONLY = re.compile(r"(?is)^\s*select\b")

def _is_select_only(sql: str) -> bool:
    s = sql.strip()
    if ";" in s:
        return False
    return bool(_SELECT_ONLY.match(s))


# ====================== Tool 1: Schema discovery ============================

class MySQLSchemaTool(BaseTool):
    name: str = "mysql_schema_tool"
    description: str = (
        "Discover MySQL schema. "
        "Input: empty (lists all tables) OR a comma-separated list of tables."
    )

    def _run(self, table_list: str = "") -> str:
        db = _get_db()
        engine = _get_engine()
        with engine.connect() as conn:
            if not table_list.strip():
                tables = [r[0] for r in conn.execute(text("SHOW TABLES")).fetchall()]
            else:
                tables = [t.strip() for t in table_list.split(",") if t.strip()]
            if not tables:
                return "No tables found."

            lines: List[str] = []
            for t in tables:
                try:
                    rows = conn.execute(text(f"DESCRIBE `{t}`")).fetchall()
                except Exception as e:
                    lines.append(f"# {t}\nERROR: {e}")
                    continue
                lines.append(f"# {t}")
                for field, col_type, nullable, key, default, extra in rows:
                    lines.append(
                        f"- {field}: {col_type}, NULL={nullable}, KEY={key}, "
                        f"DEFAULT={default}, EXTRA={extra}"
                    )
        # Append a compact database-level table info snapshot (first few lines)
        try:
            snapshot = db.get_table_info()
            if snapshot:
                head = "\n".join(snapshot.splitlines()[:20])
                lines.append("\n# table_info snapshot (truncated):\n" + head)
        except Exception:
            pass
        return "\n".join(lines)

    async def _arun(self, *args: Any, **kwargs: Any) -> str:
        raise NotImplementedError


# ====================== Tool 2: Natural language -> SQL =====================
class MySQLNL2SQLTool(BaseTool):
    name: str = "mysql_nl2sql_tool"
    description: str = (
        "Generate a single, safe MySQL SELECT query from a natural-language question. "
        "Input: the user question. Output: SQL only."
    )
    llm: Any = None  # ChatOpenAI injected at construction

    def _run(self, question: str) -> str:
        if not question or not question.strip():
            return "Please provide a question."
        db = _get_db()

        system = (
            "You are a senior data analyst. Produce ONE valid MySQL SELECT query that "
            "answers the question using only existing tables/columns. "
            "No comments or prose. No DDL/DML. SELECT-only."
        )

        # IMPORTANT: Use input variables: input, table_info, top_k
        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", system),
                (
                    "human",
                    "Database schema:\n{table_info}\n\n"
                    "Question:\n{input}\n\n"
                    "Constraints:\n"
                    "- Only one SELECT statement\n"
                    "- No semicolons\n"
                    "- MySQL dialect\n\n"
                    "SQL:"
                ),
            ]
        ).partial(top_k="5")  # or set via k=5 below

        # Either set k here or via .partial(top_k="...")
        chain = create_sql_query_chain(self.llm, db, prompt=prompt, k=5)

        # IMPORTANT: pass {input}, not {question}; table_info is injected by the chain.
        sql = chain.invoke({"input": question}).strip()

        # Safety: enforce a single SELECT without semicolons
        sql = re.sub(r";.*$", "", sql, flags=re.S).strip()
        if not _is_select_only(sql):
            return "Failed to produce a safe single SELECT statement. Try rephrasing."
        return sql

class MySQLNL2SQLTool_1(BaseTool):
    name: str = "mysql_nl2sql_tool"
    description: str = (
        "Generate a single, safe MySQL SELECT query from a natural-language question. "
        "Input: the user question. Output: SQL only."
    )
    llm: Any = None  # ChatOpenAI injected at construction

    def _run(self, question: str) -> str:
        if not question or not question.strip():
            return "Please provide a question."
        db = _get_db()

        system = (
            "You are a senior data analyst. Produce ONE valid MySQL SELECT query that "
            "answers the question using only existing tables/columns. "
            "No comments or prose. No DDL/DML. SELECT-only."
        )
        prompt = ChatPromptTemplate.from_messages(
            [
                ("system", system),
                ("human", "Database schema:\n{schema}\n\nQuestion:\n{question}\n\nSQL:"),
            ]
        )

        #chain = create_sql_query_chain(self.llm, db, prompt=prompt)
        #sql = chain.invoke({"question": question, "schema": db.get_table_info()}).strip()
        chain = create_sql_query_chain(self.llm, _get_db(), k=5)
        sql = chain.invoke({"input": question}).strip()

        # Safety
        sql = re.sub(r";.*$", "", sql, flags=re.S).strip()
        if not _is_select_only(sql):
            return "Failed to produce a safe single SELECT statement. Try rephrasing."
        return sql

    async def _arun(self, *args: Any, **kwargs: Any) -> str:
        raise NotImplementedError


# ====================== Tool 3: Execute SQL (read-only) =====================

class MySQLQueryExecTool(BaseTool):
    name: str = "mysql_query_exec_tool"
    description: str = (
        "Execute a single SELECT query against MySQL and return rows (capped). "
        "Input must be ONE SELECT statement, no semicolons."
    )

    def _run(self, sql: str) -> str:
        if not _is_select_only(sql):
            return "Only a single SELECT statement without ';' is allowed."
        row_cap = int(os.getenv("MYSQL_MAX_ROWS", "200"))
        engine = _get_engine()
        with engine.connect() as conn:
            result = conn.execute(text(sql))
            rows = result.fetchmany(row_cap + 1)
            headers = list(result.keys())
        truncated = len(rows) > row_cap
        rows = rows[:row_cap]

        out = []
        header_line = " | ".join(map(str, headers))
        out.append(header_line)
        out.append("-" * max(3, len(header_line)))
        for r in rows:
            out.append(" | ".join("" if v is None else str(v) for v in r))
        if truncated:
            out.append(f"...(truncated at {row_cap} rows)")
        return "\n".join(out)

    async def _arun(self, *args: Any, **kwargs: Any) -> str:
        raise NotImplementedError


# ====================== Agent builder =======================================

def build_web_agent(model_name: Optional[str] = None):
    """Create a LangGraph ReAct agent with DuckDuckGo + MySQL tools."""
    llm_id = model_name or os.getenv("OPENAI_MODEL", "gpt-4o-mini")
    llm = ChatOpenAI(model=llm_id, temperature=0)

    ddg = DuckDuckGoSearchRun()
    mysql_schema = MySQLSchemaTool()
    mysql_nl2sql = MySQLNL2SQLTool(llm=llm)
    mysql_exec = MySQLQueryExecTool()

    tools = [ddg, mysql_schema, mysql_nl2sql, mysql_exec]
    #tools = [mysql_schema, mysql_nl2sql, mysql_exec]
    agent = create_react_agent(llm, tools)
    return agent


# ====================== Minimal demo ========================================

if __name__ == "__main__":
    # Example: stream an answer for a web question (your original flow)
    agent = build_web_agent()
    print("[demo] Running…")
    start = time.time()
    #user_input="provide description of each table in the database mlflow_db"
    #user_input="provide me description & counts of all columns of the table datasets and its signifcance"
    #user_input="provide me counts of each table"
    #user_input="provide me description of tags table"
    #user_input="provide me description of params table"
    ##user_input="provide me best metrics and its value for experiment id 1"
    ##user_input="provide me count of runs for experiment_id 1"
    ##user_input="what are the distinct metrics and its maximum values for experiment id 1"
    ##user_input="This is a mlflow database, there are experiments stored. provide me the distinct tags from the table tags for experiment id 1"
    #user_input=" provide me  distinct metric and its value each experiment"
    user_input="provide me counts of each table"
    for event in agent.stream({"messages": [("user", user_input)] }):
        for node, data in event.items():
            msgs = data.get("messages") if isinstance(data, dict) else None
            if msgs:
                last = msgs[-1]
                role = getattr(last, "type", getattr(last, "role", ""))
                content = getattr(last, "content", str(last))
                if content:
                    print(f"[{node}] {content}")
    #print(agent.invoke("provide me description & counts of all columns of the table datasets and its signifcance"))
    print(f"[demo] Done in {time.time()-start:.2f}s")

    # You can also drive the MySQL tools explicitly:
    # 1) print(MySQLSchemaTool().invoke(""))                 # list all tables
    # 2) sql = MySQLNL2SQLTool(llm=ChatOpenAI(model="gpt-4o-mini", temperature=0)).invoke(
    #        "top 10 customers by total order value in 2024")
    #    print(sql)
    # 3) print(MySQLQueryExecTool().invoke(sql))


C:\Users\surya.adatravu\AppData\Local\Temp\ipykernel_19332\382640105.py:249: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools)


[demo] Running…
[tools] # alembic_version
- version_num: varchar(32), NULL=NO, KEY=PRI, DEFAULT=None, EXTRA=
# datasets
- dataset_uuid: varchar(36), NULL=NO, KEY=MUL, DEFAULT=None, EXTRA=
- experiment_id: int(11), NULL=NO, KEY=PRI, DEFAULT=None, EXTRA=
- name: varchar(500), NULL=NO, KEY=PRI, DEFAULT=None, EXTRA=
- digest: varchar(36), NULL=NO, KEY=PRI, DEFAULT=None, EXTRA=
- dataset_source_type: varchar(36), NULL=NO, KEY=, DEFAULT=None, EXTRA=
- dataset_source: text, NULL=NO, KEY=, DEFAULT=None, EXTRA=
- dataset_schema: mediumtext, NULL=YES, KEY=, DEFAULT=None, EXTRA=
- dataset_profile: mediumtext, NULL=YES, KEY=, DEFAULT=None, EXTRA=
# experiment_tags
- key: varchar(250), NULL=NO, KEY=PRI, DEFAULT=None, EXTRA=
- value: varchar(5000), NULL=YES, KEY=, DEFAULT=None, EXTRA=
- experiment_id: int(11), NULL=NO, KEY=PRI, DEFAULT=None, EXTRA=
# experiments
- experiment_id: int(11), NULL=NO, KEY=PRI, DEFAULT=None, EXTRA=auto_increment
- name: varchar(256), NULL=NO, KEY=UNI, DEFAULT=None, EXTRA=

In [2]:
pip show langchain

Name: langchain
Version: 1.0.3
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: C:\Users\surya.adatravu\AppData\Local\anaconda3\envs\r1\Lib\site-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
Note: you may need to restart the kernel to use updated packages.


In [3]:
!pip install langchain-classic